In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

# Telemetry #

In [2]:
df_tel = pd.read_csv('Behavioral_4HZ/labeled_combined_behavioral_4HZ.csv',
                      usecols=['heading[deg]', 'pitch[deg]', 'roll[deg]', 'speed[m/s]', 'rpm', 'gear','user', 'session', 'label'])

### Imputing Missing Telemetry Data for User A_A with Sequential Session Transfer Based on Label Similarity ###

In [ ]:
# Columns to fill gaps
telemetry_columns = ["heading[deg]", "pitch[deg]", "roll[deg]", "speed[m/s]", "rpm", "gear"]

# Create a mask for the missing data (User A, session Alert)
missing_mask = (df_tel['user'] == 'A') & (df_tel['session'] == 'Alert')

# Get the target label for User A, session Alert
target_label = df_tel.loc[missing_mask, 'label'].iloc[0]

# Filter data with the same label, excluding User A's Alert session
similar_data = df_tel[(df_tel['label'] == target_label) & ~(df_tel['user'] == 'A') & 
                     ~df_tel[telemetry_columns].isna().any(axis=1)]

# Group by user and session to get complete sessions
session_groups = similar_data.groupby(['user', 'session'])

# Count number of rows to fill
n_missing_rows = missing_mask.sum()

# Function to select and transfer a session
def transfer_session(session_data, n_rows):
    # If session has fewer rows, repeat the sequence
    if len(session_data) < n_rows:
        repeat_factor = int(np.ceil(n_rows / len(session_data)))
        repeated_data = pd.concat([session_data] * repeat_factor, ignore_index=True)
        return repeated_data[telemetry_columns].iloc[:n_rows].reset_index(drop=True)
    # If session has enough or more rows, truncate
    return session_data[telemetry_columns].iloc[:n_rows].reset_index(drop=True)

# Choose a session to transfer
available_sessions = list(session_groups.groups.keys())
if available_sessions:
    # For simplicity, select the first session (could be randomized or based on similarity)
    selected_session = available_sessions[0]  # Or use np.random.choice() for random selection
    selected_data = session_groups.get_group(selected_session)
    
    # Transfer the session's time series data
    imputed_data = transfer_session(selected_data, n_missing_rows)
    
    # Fill missing values with the sequential session data
    df_tel.loc[missing_mask, telemetry_columns] = imputed_data.values
    
    print(f"Filled missing data using session from user {selected_session[0]}, "
          f"session {selected_session[1]}")
else:
    print("No complete sessions found with the same label for imputation")
    # Fall back to interpolation if no sessions available
    df_tel.loc[missing_mask, telemetry_columns] = df_tel[telemetry_columns].interpolate(
        method='linear', limit_direction='both'
    ).loc[missing_mask]

### Interpolate Missing Values in Behavioral Data ###

In [4]:
# Select only numerical columns for interpolation
numerical_columns = df_tel.select_dtypes(include=['number']).columns

# Group the data by 'user' and 'session' and apply interpolation separately for each group
df_tel[numerical_columns] = df_tel.groupby(['user', 'session'])[numerical_columns].apply(
    lambda group: group.interpolate(method='linear', axis=0, limit_direction='forward')
).reset_index(drop=True, level=['user', 'session'])



In [5]:
# Create necessary directories if they don't exist
os.makedirs("4HZ_Individual_Data", exist_ok=True)

# Optionally save the clean data to a new CSV file
df_tel.to_csv('4HZ_Individual_Data/final_telemetry_4HZ.csv', index=False)

In [ ]:
df_tel['DrowsinessLevel'] = df_tel['label'].apply(lambda x: 0 if x < 4 else 1 if x<7 else 2)
df_tel = df_tel.drop(columns=['label'])

### Applying SVM for classification (Telemetry) ###

In [ ]:
%%time
# --- Setup Cross-validation ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Separate features and target
X = df_tel.drop(columns=['DrowsinessLevel', 'session', 'user'])
y = df_tel['DrowsinessLevel']

# Apply Min-Max normalization
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# --- PCA Feature Selection ---
# PCA to retain a specific amount of variance
pca = PCA(n_components=0.90) 
X_pca = pca.fit_transform(X_normalized)

# Print the number of components selected and the explained variance ratio
print(f"Number of principal components selected: {X_pca.shape[1]}")
print(f"Explained variance ratio per component: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_)}")

# --- SVM ---
svm_clf = SVC(kernel='rbf', C=0.1, gamma='auto', cache_size=200,class_weight='balanced', random_state=42) 
svm_cv_scores = cross_val_score(svm_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("SVM CV Accuracy: {:.2f} (+/- {:.2f})".format(svm_cv_scores.mean(), svm_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(svm_clf, X_pca, y, cv=cv, n_jobs=-1)
print("SVM CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("SVM Normalized Confusion Matrix (5-Fold CV)")
plt.show()

# --- Random Forest ---
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_cv_scores = cross_val_score(rf_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("RF CV Accuracy: {:.2f} (+/- {:.2f})".format(rf_cv_scores.mean(), rf_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(rf_clf, X_pca, y, cv=cv, n_jobs=-1)
print("RF CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("Random Forest Normalized Confusion Matrix (5-Fold CV)")
plt.show()


### Grip Pressure ###

In [11]:
df_grip = pd.read_csv('Behavioral_4HZ/labeled_combined_behavioral_4HZ.csv',
                      usecols=['LGP', 'RGP','user', 'session', 'label'])

In [12]:
# Optionally save the clean data to a new CSV file
df_grip.to_csv('4HZ_Individual_Data/final_grip_4HZ.csv', index=False)

### Interpolate Missing Values in Grip Data ###

In [13]:
# Select only numerical columns for interpolation
numerical_columns = df_grip.select_dtypes(include=['number']).columns

# Group the data by 'user' and 'session' and apply interpolation separately for each group
df_grip[numerical_columns] = df_grip.groupby(['user', 'session'])[numerical_columns].apply(
    lambda group: group.interpolate(method='linear', axis=0, limit_direction='forward')
).reset_index(drop=True, level=['user', 'session'])

In [14]:
df_grip['DrowsinessLevel'] = df_grip['label'].apply(lambda x: 0 if x < 4 else 1 if x<7 else 2)
df_grip = df_grip.drop(columns=['label'])

### Applying SVM for classification (Grip Pressure) ###

In [ ]:
%%time
# --- Setup Cross-validation ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Separate features and target
X = df_grip.drop(columns=['DrowsinessLevel', 'session', 'user'])
y = df_grip['DrowsinessLevel']

# Apply Min-Max normalization
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# --- PCA Feature Selection ---
# PCA to retain a specific amount of variance
pca = PCA(n_components=0.90) 
X_pca = pca.fit_transform(X_normalized)

# Print the number of components selected and the explained variance ratio
print(f"Number of principal components selected: {X_pca.shape[1]}")
print(f"Explained variance ratio per component: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_)}")

# --- SVM ---
svm_clf = SVC(kernel='rbf', C=0.1, gamma='auto', cache_size=200,class_weight='balanced', random_state=42) 
svm_cv_scores = cross_val_score(svm_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("SVM CV Accuracy: {:.2f} (+/- {:.2f})".format(svm_cv_scores.mean(), svm_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(svm_clf, X_pca, y, cv=cv, n_jobs=-1)
print("SVM CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("SVM Normalized Confusion Matrix (5-Fold CV)")
plt.show()

# --- Random Forest ---
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_cv_scores = cross_val_score(rf_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("RF CV Accuracy: {:.2f} (+/- {:.2f})".format(rf_cv_scores.mean(), rf_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(rf_clf, X_pca, y, cv=cv, n_jobs=-1)
print("RF CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("Random Forest Normalized Confusion Matrix (5-Fold CV)")
plt.show()


### Pose Landmarks ###

In [16]:
# Load the DataFrame first
df_pose = pd.read_csv('Behavioral_4HZ/labeled_combined_behavioral_4HZ.csv')

# Dynamically select columns that contain "_x", "_y", or "_z"
selected_columns = [col for col in df_pose.columns if '_x' in col or '_y' in col or '_z' in col]

# Add 'user', 'session', and 'label' columns to the selected columns
selected_columns += ['user', 'session', 'label']

# Load the relevant columns based on the selection
df_pose = pd.read_csv('Behavioral_4HZ/labeled_combined_behavioral_4HZ.csv',
                      usecols=selected_columns)

### Interpolate Missing Values in Pose Data ###

In [17]:
# Select only numerical columns for interpolation
numerical_columns = df_pose.select_dtypes(include=['number']).columns

# Group the data by 'user' and 'session' and apply interpolation separately for each group
df_pose[numerical_columns] = df_pose.groupby(['user', 'session'])[numerical_columns].apply(
    lambda group: group.interpolate(method='linear', axis=0, limit_direction='forward')
).reset_index(drop=True, level=['user', 'session'])

In [18]:
# Optionally save the clean data to a new CSV file
df_pose.to_csv('4HZ_Individual_Data/final_pose_4HZ.csv', index=False)

In [ ]:
df_pose['DrowsinessLevel'] = df_pose['label'].apply(lambda x: 0 if x < 4 else 1 if x<7 else 2)
df_pose = df_pose.drop(columns=['label'])

### Applying SVM for classification (Pose Landmarks) ###

In [ ]:
%%time
# --- Setup Cross-validation ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Separate features and target
X = df_pose.drop(columns=['DrowsinessLevel', 'session', 'user'])
y = df_pose['DrowsinessLevel']

# Apply Min-Max normalization
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# --- PCA Feature Selection ---
# PCA to retain a specific amount of variance
pca = PCA(n_components=0.90)  
X_pca = pca.fit_transform(X_normalized)

# Print the number of components selected and the explained variance ratio
print(f"Number of principal components selected: {X_pca.shape[1]}")
print(f"Explained variance ratio per component: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_)}")

# --- SVM ---
svm_clf = SVC(kernel='rbf', C=0.1, gamma='auto', cache_size=200,class_weight='balanced', random_state=42) 
svm_cv_scores = cross_val_score(svm_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("SVM CV Accuracy: {:.2f} (+/- {:.2f})".format(svm_cv_scores.mean(), svm_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(svm_clf, X_pca, y, cv=cv, n_jobs=-1)
print("SVM CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("SVM Normalized Confusion Matrix (5-Fold CV)")
plt.show()

# --- Random Forest ---
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_cv_scores = cross_val_score(rf_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("RF CV Accuracy: {:.2f} (+/- {:.2f})".format(rf_cv_scores.mean(), rf_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(rf_clf, X_pca, y, cv=cv, n_jobs=-1)
print("RF CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("Random Forest Normalized Confusion Matrix (5-Fold CV)")
plt.show()
